In [ ]:
#| export machine_learning.note_linking

from datasets import Dataset
import random
from typing import Literal, Optional, TypedDict

from trouver.machine_learning.note_data import (
    NoteLinkEnum, InfoNoteData, NotatNoteData, _update_dict
    )


# Notation Summarization using `NoteData` classes 

**The functions here are not yet useful**

The functions above surrounding the `NoteData` classes should actually be close to providing the means for gathering data for other ML tasks, such as the summarization task (thus far provided by `25_machine_learning.notation.summarization.ipynb`) and the definition naming task (thus far provided by `35_machine_learning.definition_and_notation_naming.ipynb`), and even improve upon them by providing contextual data (given by the notes linked by a given note).

In [ ]:
#| export machine_learning.note_linking
class SummarizationDataPoint(TypedDict):
    input: str
    output: str
    notat_note_name: str

In [ ]:
#| export machine_learning.note_linking
def summarization_data(
        notat_note_data_point: NotatNoteData,
        info_note_data: dict[str, InfoNoteData], # For getting data from the linked notes.
        notat_note_data: dict[str, NotatNoteData], # For getting data from the linked notes.
        format: Literal['bert', 't5'],
        augmentation: Optional[Literal['high', 'mid', 'low']] = None,
        ) -> SummarizationDataPoint:
    """
    Compile the summarization data from a `NotatNoteData` . 

    `notat_note_data_point` must have a nonblank value for its `note_content` attribute.

    The summarization data consists of the notation note's main note content and (optionally)
    the position data (which is the position data of the main note) along with (also optionally)
    the content and/or position data of info notes that either the notation note or its
    main note depend on (via the `NoteLinkEnum.NOTAT_TO_INFO_VIA_NOTAT` or 
    `NoteLinkEnum.INFO_TO_INFO_IN_CONTENT` enum items).

    The augmentations are not applied to `notat_note_data_point` but are rather applied to
    (copies of) the info notes that the notation note or its main note depends on. Use
    `augment_notat_note_data_for_summarization` to augment that `NotatNoteData` object.

    **Raises**
    - `ValueError`
        - if `notat_note_data_point.note_content` is not a nonblank `str`.
    """
    content = notat_note_data_point.note_content
    if content is None or content.strip() == '':
        raise ValueError(f"Expected `notat_note_data_point.content` to be a non blank string but was {notat_note_data_point.content}. The relevant notation note name is {notat_note_data_point.note_name}.")
    # Temporarily blank out the `note_content` attribute for the `.data_string` method
    # To exclude whatever is in the `note_content` attribute.
    notat_note_data_point.note_content = None
    notat_note_string = notat_note_data_point.data_string(format)
    notat_note_data_point.note_content = content

    info_note_names_to_consider: set[str] = set()
    for relied_note_name, link_types in notat_note_data_point.directly_linked_notes.items():
        if NoteLinkEnum.NOTAT_TO_INFO_VIA_NOTAT in link_types:
            info_note_names_to_consider.add(relied_note_name)
    if (notat_note_data_point.main_note
            and notat_note_data_point.main_note in info_note_data):
        main_note_data_point = info_note_data[notat_note_data_point.main_note]
        for relied_note_name, link_types in main_note_data_point.directly_linked_notes.items():
            if NoteLinkEnum.INFO_TO_INFO_IN_CONTENT in link_types:
                info_note_names_to_consider.add(relied_note_name)

    parts: list[str] = [notat_note_string]
    for relied_note_name in list(info_note_names_to_consider):
        if not relied_note_name in info_note_data:
            continue
        relied_note_data_point = info_note_data[relied_note_name]
        relied_note_data_point = relied_note_data_point.deepcopy()
        if augmentation:
            erase_position_metadata = _erase_position_metadata(augmentation)
            relied_note_data_point.randomly_modify(augmentation, erase_position_metadata)
        parts.append(relied_note_data_point.data_string(format))

    sep_str = '\n\n[SEP]\n\n' if format == 'bert' else '\n\n</s>\n\n'
    input = sep_str.join(parts)
    output = notat_note_data_point.note_content
    return SummarizationDataPoint(
        input=input, output=output,
        notat_note_name=notat_note_data_point.note_name)


In [ ]:
#| export machine_learning.note_linking
def augment_notat_note_data_for_summarization(
        notat_note_data_point: NotatNoteData,
        augmentation: Literal['high', 'mid' ,'low'],
        info_note_data: dict[str, InfoNoteData], # For getting data from the linked notes.
        modify_links: bool = True, # If `True`, randomly modify the linking data from that of `notat_note_data_point`
        ) -> NotatNoteData:
    """
    Return a modified copy of `notat_note_data` augmented for providing
    notation summarization data.

    The `note_content` attribute of the outputted copy should not be modified
    as it serves as the output of the training data.
    """
    notat_note_data_copy = notat_note_data_point.deepcopy()
    erase_position_metadata = _erase_position_metadata(augmentation)
    content = notat_note_data_copy.note_content
    notat_note_data_copy.randomly_modify(augmentation, erase_position_metadata)
    notat_note_data_copy.note_content = content

    if modify_links:
        if augmentation == 'high':
            num_rand_info_note_data_to_add = 3
            key_deletion_prob = 0.10
        elif augmentation == 'mid':
            num_rand_info_note_data_to_add = 2
            key_deletion_prob = 0.05
        elif augmentation == 'low':
            num_rand_info_note_data_to_add = 1
            key_deletion_prob = 0.02
        else:
            num_rand_info_note_data_to_add = 0
            key_deletion_prob = 0
        # Delete "links" at random.
        keys = list(notat_note_data_copy.directly_linked_notes)
        for key in keys:
            if random.random() < key_deletion_prob:
                notat_note_data_copy.directly_linked_notes.pop(key)
        # Add "links" to info notes at random.
        random_info_note_names_to_add = random.choices(
            list(info_note_data), k=min(len(info_note_data), num_rand_info_note_data_to_add))
        for random_info_note_name in random_info_note_names_to_add:
            _update_dict(
                notat_note_data_copy.directly_linked_notes, 
                random_info_note_name,
                NoteLinkEnum.NOTAT_TO_INFO_VIA_NOTAT)
    return notat_note_data_copy

In [ ]:
#| export machine_learning.note_linking
def notat_note_data_admissible_for_summarization_data(
        notat_note_data_point: NotatNoteData
        ) -> bool:  # `True` if the notation note data does not have the `_auto/notation_summary` tag, and the content of the notation note is essentially note blank.
    if notat_note_data_point.tags and '_auto/notation_summary' in notat_note_data_point.tags:
        return False
    if notat_note_data_point.note_content is None:
        return False 
    return bool(notat_note_data_point.note_content.strip())

In [ ]:

# for name, data_point in notat_note_data.items():
#     if notat_note_data_admissible_for_summarization_data(data_point):
#         print(name)
#         break

# summ_data = summarization_data(data_point, info_note_data, notat_note_data, 'bert')
# print(summ_data['input'])

In [ ]:
# notat_note_data_admissible_for_summarization_data(notat_note_data['achter_pries_imht_notation_T_bar_gamma_smooth_trielliptic_curves_with_inertia_type'])

In [ ]:
#| export machine_learning.note_linking
def _add_augmented_data_points(
        info_note_data: dict[str, InfoNoteData],
        notat_note_data: dict[str, NotatNoteData],
        format: Literal['bert', 't5'],
        data_point: NotatNoteData,
        dict_data: list[SummarizationDataPoint],
        ):
    """
    Helper function to `summarization_dataset_from_note_data`.
    """
    data_point_without_links = data_point.deepcopy()
    data_point_without_links.directly_linked_notes = {}
    for augmentation in ['low', 'mid', 'high']:
        for modify_links, base_data_point in zip([True, False], [data_point, data_point_without_links]):
            aug_data_point: NotatNoteData = augment_notat_note_data_for_summarization(
                base_data_point, augmentation, info_note_data, modify_links)
            dict_data.append(summarization_data(
                aug_data_point, info_note_data, notat_note_data, format, augmentation))
    

In [ ]:
#| export machine_learning.note_linking
def summarization_dataset_from_note_data(
        info_note_data: dict[str, InfoNoteData],
        notat_note_data: dict[str, NotatNoteData],
        augment: bool,
        format: Literal['bert', 't5'],
        ) -> Dataset:
    admissible_notat_note_names: list[str] = []
    for notat_note_name, data_point in notat_note_data.items():
        if notat_note_data_admissible_for_summarization_data(data_point):
            admissible_notat_note_names.append(notat_note_name)
    dict_data: list[SummarizationDataPoint] = []
    for admissible_notat_note_name in admissible_notat_note_names:
        data_point: NotatNoteData = notat_note_data[admissible_notat_note_name]
        dict_data.append(summarization_data(
            data_point, info_note_data, notat_note_data, format, augmentation=None))
        if not augment:
            continue
        _add_augmented_data_points(info_note_data, notat_note_data, format, data_point, dict_data)
    return Dataset.from_list(dict_data)